# Off-Platform Project: Favourites
This off-platform project uses the ```random_tweets``` dataset to predict whether a tweet will be favourited.

In [1]:
import pandas as pd
import numpy as np

## Explore the Dataset
Let's start by taking a look at the data. After importing ```random_tweets.json```, find the following information that we can use to gauge a tweet's potential to be favourited:
* The number of tweets.
* The columns, or features, of a tweet.

In [2]:
all_tweets = pd.read_json('random_tweets.json', lines=True)
print('Here\'s a preview of the `random_tweets` dataset:\n', all_tweets.head())

Here's a preview of the `random_tweets` dataset:
                  created_at                   id               id_str  \
0 2018-07-31 13:34:40+00:00  1024287229525598210  1024287229525598210   
1 2018-07-31 13:34:40+00:00  1024287229512953856  1024287229512953856   
2 2018-07-31 13:34:40+00:00  1024287229504569344  1024287229504569344   
3 2018-07-31 13:34:40+00:00  1024287229496029190  1024287229496029190   
4 2018-07-31 13:34:40+00:00  1024287229492031490  1024287229492031490   

                                                text  truncated  \
0  RT @KWWLStormTrack7: We are more than a month ...      False   
1  @hail_ee23 Thanks love its just the feeling of...      False   
2  RT @TransMediaWatch: Pink News has more on the...      False   
3  RT @realDonaldTrump: One of the reasons we nee...      False   
4  RT @First5App: This hearing of His Word doesn’...      False   

                                            entities  \
0  {'hashtags': [], 'symbols': [], 'user_mentions...

In [3]:
print(f'There are {len(all_tweets)} tweets in this dataset.')
print('This dataset also has the following variables:\n', all_tweets.columns)

There are 11099 tweets in this dataset.
This dataset also has the following variables:
 Index(['created_at', 'id', 'id_str', 'text', 'truncated', 'entities',
       'metadata', 'source', 'in_reply_to_status_id',
       'in_reply_to_status_id_str', 'in_reply_to_user_id',
       'in_reply_to_user_id_str', 'in_reply_to_screen_name', 'user', 'geo',
       'coordinates', 'place', 'contributors', 'retweeted_status',
       'is_quote_status', 'retweet_count', 'favourite_count', 'favourited',
       'retweeted', 'lang', 'possibly_sensitive', 'quoted_status_id',
       'quoted_status_id_str', 'extended_entities', 'quoted_status',
       'withheld_in_countries'],
      dtype='str')


**Variables to Use:**
* *Outcome:* ```was_favourited``` (derive from ```favourite_count```)
* *Features:*
  * Author reach: ```log_followers```, ```log_friends```, ```verified``` (derive from ```user```)
  * Tweet composition: ```tweet_length```, ```hashtag_count```, ```mention_count```, ```url_count```, ```has_media``` (derive from ```entities```)
  * Conversation context: ```is_reply``` (derive from ```in_reply_to_user_id```), ```is_quote``` (derive from ```is_quote_status```)

**Machine Learning Models to Use:**
* Naïve Bayes (on tweet composition)
* Logistic regression (on author reach, tweet composition, and context)

## Prepare Data for Analysis

### Cleaning

In [4]:
# Remove tweets with duplicate IDs
cleaned_tweets = all_tweets.drop_duplicates(subset=['id']).copy()

In [5]:
# Remove tweets lest they contain empty values
cleaned_tweets = cleaned_tweets.dropna(
    subset=[
        'id',
        'text',
        'user',
        'entities',
        'favourite_count'
    ]
).copy()

In [6]:
# Exclude retweets 
original_tweets = cleaned_tweets[
    all_tweets['retweeted_status'].isna()
].copy()

### Defining Variables

In [7]:
original_tweets['was_favourited'] = (
    original_tweets['favourite_count']>0
).astype(int)

In [8]:
# Author reach
original_tweets['log_followers'] = original_tweets['user'].apply(
    lambda user: np.log1p(user.get('followers_count', 0))
)
original_tweets['log_friends'] = original_tweets['user'].apply(
    lambda user: np.log1p(user.get('friends_count', 0))
)
original_tweets['verified'] = original_tweets['user'].apply(
    lambda user: int(user.get('verified', False))
)

In [9]:
# Tweet composition
original_tweets['tweet_length'] = original_tweets['text'].str.len()
original_tweets['hashtag_count'] = original_tweets['entities'].apply(
    lambda entities: len(entities.get('hashtags', []))
)
original_tweets['mention_count'] = original_tweets['entities'].apply(
    lambda entities: len(entities.get('user_mentions', []))
)
original_tweets['url_count'] = original_tweets['entities'].apply(
    lambda entities: len(entities.get('urls', []))
)
original_tweets['has_media'] = original_tweets['entities'].apply(
    lambda entities: int(len(entities.get('media', [])) > 0)
)

In [10]:
# Conversation context
original_tweets['is_reply'] = (
    original_tweets['in_reply_to_user_id'].notna().astype(int)
)
original_tweets['is_quote'] = (
    original_tweets['is_quote_status'].astype(int)
)

## The Training-Testing Split
Now that we have our desired variables (see above), it's time to break them up into training and testing sets.

**Input:**
* The dataset (```original_tweets```)
* The label (```was_favourited```)

**Output:**
* The training data and labels
* The testing data and labels

**Method:** ```train_test_split()``` from ```scikit-learn```

In [27]:
labels = original_tweets['was_favourited'] # 0 means 'No', and 1 means 'Yes'
print(labels.value_counts())

was_favourited
0    3501
1     226
Name: count, dtype: int64


In [12]:
from sklearn.model_selection import train_test_split
training_data, testing_data, training_labels, testing_labels = train_test_split(original_tweets,
                                                                                labels,
                                                                                test_size=0.2,
                                                                                random_state=1,
                                                                                stratify=labels)

## Set a Baseline
As you can see above, favourited tweets make up only a small fraction of the dataset. As a result, accuracy may not be a useful metric of model performance. A majority-class baseline will show what happens when every tweet is classified as the most common outcome: not favourited. In order to be able to accurately predict a tweet's favourited status, a model's performance must outperform this baseline.

In [23]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score

In [24]:
baseline = DummyClassifier(strategy='most_frequent')
baseline.fit(training_data[['tweet_length']], training_labels)
baseline_predictions = baseline.predict(testing_data[['tweet_length']])

In [25]:
print(f'Baseline accuracy: {accuracy_score(testing_labels, baseline_predictions)*100:.4f}%')
print(f'Baseline confusion matrix:\n{confusion_matrix(testing_labels, baseline_predictions)}')
print(f'Baseline precision: {precision_score(testing_labels, baseline_predictions, zero_division=0)*100:.4f}%')
print(f'Baseline recall: {recall_score(testing_labels, baseline_predictions, zero_division=0)*100:.4f}%')
print(f'Baseline F1-score: {f1_score(testing_labels, baseline_predictions, zero_division=0)*100:.4f}%')

Baseline accuracy: 93.9678%
Baseline confusion matrix:
[[701   0]
 [ 45   0]]
Baseline precision: 0.0000%
Baseline recall: 0.0000%
Baseline F1-score: 0.0000%


## Train, Test, and Evaluate the Machine Learning Models

### The Naïve Bayes Classifier
**Purpose:** uses words in text to determine whether the tweet was favourited

In [13]:
# 1. Extract text from tweets
training_nb, testing_nb = training_data['text'], testing_data['text']

In [14]:
# 2. Count the number of words from text
from sklearn.feature_extraction.text import CountVectorizer
counter = CountVectorizer()
training_counts = counter.fit_transform(training_nb)
testing_counts = counter.transform(testing_nb)

In [15]:
# 3. Train and test the model
from sklearn.naive_bayes import MultinomialNB
nb = MultinomialNB()
nb.fit(training_counts, training_labels)
nb_predictions = nb.predict(testing_counts)

In [20]:
# 4. Evaluate the model
print(f'Naïve Bayes model accuracy: {accuracy_score(testing_labels, nb_predictions)*100:.4f}%')
print(f'Naïve Bayes confusion matrix:\n{confusion_matrix(testing_labels, nb_predictions)}')

Naïve Bayes model accuracy: 93.9678%
Naïve Bayes confusion matrix:
[[701   0]
 [ 45   0]]


**Verdict:** This model behaves exactly the same as the baseline, meaning it was unable to distinguish favourited tweets from unfavourited ones. Therefore, tweet wording alone did not register enough for this Naïve Bayes model to identify the minority class.

### The Logistic Regression Model
**Purpose:** determines whether a tweet was favourited based on author reach, tweet composition, and context

In [17]:
# 1. Extract training/testing data from the features below
logistic_features = [
    'log_followers',
    'log_friends',
    'verified',
    'tweet_length',
    'hashtag_count',
    'mention_count',
    'url_count',
    'has_media',
    'is_reply',
    'is_quote',
]
training_lr = training_data[logistic_features]
testing_lr = testing_data[logistic_features]

In [18]:
# 2. Train and test the model
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression(
    max_iter=10_000,
    class_weight='balanced',
    solver='liblinear',
    random_state=1
)
lr.fit(training_lr, training_labels)
lr_predictions = lr.predict(testing_lr)

In [19]:
# 3. Evaluate the model
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score # because accuracy can be misleading
# 3.1. See how many tweets were correctly (or incorrectly) identified
print(f'Logistic regression model confusion matrix:\n{confusion_matrix(testing_labels, lr_predictions)}')
# 3.2. Gauge our model's ability to identify true positives
print(f'Logistic regression model precision: {precision_score(testing_labels, lr_predictions)*100:.4f}%')
# 3.3. See how well our model avoids making false negatives
print(f'Logistic regression model recall: {recall_score(testing_labels, lr_predictions)*100:.4f}%')
# 3.4. See how well our model handles imbalanced data
print(f'Logistic regression model F1-score: {f1_score(testing_labels, lr_predictions)*100:.4f}%')

Logistic regression model confusion matrix:
[[464 237]
 [ 22  23]]
Logistic regression model precision: 8.8462%
Logistic regression model recall: 51.1111%
Logistic regression model F1-score: 15.0820%


**Verdict:** This model has 23 true positives compared to the baseline and Naïve Bayes models' zero, indicating it does a relatively superior job in predicting a tweet's favourite status. However, its precision is very low, meaning the class imbalance hinders its ability to correctly identify a favourited tweet. Therefore, the logistic regression model is not sufficiently dependable.

## Conclusion
Neither model is a reliable predictor of whether a tweet would be favourited. It all comes down to the inherent class imbalance &mdash; both the baseline and Naïve Bayes models accurately classified tweets as unfavourited, but both failed to find any of the favourited tweets. Not helping is that the latter model behaved exactly the same as the former, meaning that the number of words in a tweet is insufficient for identifying favourites in this dataset.

The logistic regression model, comparatively speaking, was more useful in classifying favourites because it used more features &mdash; author reach, tweet composition, and conversation context. However, its confusion matrix contained significantly more false positives than the baseline model, suggesting this particular model has trouble predicting whether a tweet is *not* a favourite. All things considered, logistic regression is not inherently superior to Naïve Bayes; rather, adding more (but not too many) available structured features to a model can produce more meaningful, precise predictions.

**Limitations:**
* Only 226 of the 3,727 original tweets were favourited to begin with.
* Features such as ```follower_count```, ```friend_count```, and ```in_reply_to_user_id``` provided some predictive signal collectively, but they were not dependable predictors in the sense that they could not distinguish favourited tweets with sufficient precision.
* Audience activity, topic relevance, and later distribution &mdash; features that could potentially provide insight into a tweet's favourited status &mdash; do not exist.

**Next Steps:**
* Use a more balanced sample, preferably one that uses just as many favourited tweets as unfavourited ones.
* Measure favourite counts after a fixed period.